# ハイパーパラメータ探索と実験比較

## 前提条件

- MLflow の基本概念（Experiment・Run・Artifact）を理解していること
- scikit-learn の基本的な使い方を知っていること
- 以下のパッケージがインストール済みであること
  - `mlflow>=2.8.0`
  - `optuna>=3.0`
  - `scikit-learn`
  - `numpy`

## 学習目標

1. Optuna と MLflow Tracking を統合し、trial ごとに Run を作成する方法を習得する
2. `mlflow.start_run(nested=True)` を使った親子 Run 構造を理解する
3. `trial.suggest_float` / `trial.suggest_int` / `trial.suggest_categorical` による探索空間の設定方法を学ぶ
4. `mlflow.search_runs()` でプログラム上から Run を検索・ソートする方法を習得する
5. ベストモデルを Model Registry に登録する手順を理解する

## セクション 1: セットアップとデータ準備

In [ ]:
import optuna
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

# 再現性のためシードを固定
np.random.seed(42)

# Optuna のログを抑制（必要に応じて INFO に変更）
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"MLflow version: {mlflow.__version__}")
print(f"Optuna version: {optuna.__version__}")

In [ ]:
# MLflow のトラッキング URI と Experiment を設定
mlflow.set_tracking_uri("../../mlruns")
EXPERIMENT_NAME = "optuna_hpo"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
# データ準備（sklearn 内蔵データセット: ダウンロード不要）
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"データセット: breast_cancer")
print(f"  全サンプル数: {X.shape[0]}")
print(f"  特徴量数: {X.shape[1]}")
print(f"  訓練データ: {X_train.shape[0]} サンプル")
print(f"  テストデータ: {X_test.shape[0]} サンプル")
print(f"  クラス: {list(load_breast_cancer().target_names)}")

## セクション 2: Optuna の目的関数定義

各 Optuna trial に対して MLflow の子 Run（`nested=True`）を作成します。

### 探索空間の設定

| パラメータ | 型 | 範囲 | Optuna API |
|-----------|-----|------|------------|
| `n_estimators` | 整数 | 50〜200 | `suggest_int` |
| `max_depth` | 整数 | 2〜10 | `suggest_int` |
| `min_samples_split` | 整数 | 2〜10 | `suggest_int` |
| `max_features` | カテゴリ | sqrt / log2 | `suggest_categorical` |

> **Note**: `suggest_float` の例として、別のパラメータを探索する場合は
> `trial.suggest_float("min_impurity_decrease", 0.0, 0.1)` のように使用します。

In [ ]:
def objective(trial):
    """Optuna の目的関数。trial ごとに MLflow の子 Run を作成する。"""
    # --- 探索空間の定義 ---
    params = {
        # suggest_int: 整数パラメータの探索
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        # suggest_categorical: カテゴリパラメータの探索
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        # suggest_float の例（ここでは固定値として使用）
        # "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 0.1),
        "random_state": 42,
    }

    # --- 子 Run（nested=True）の作成 ---
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        # パラメータをロギング
        mlflow.log_params(params)
        mlflow.log_param("trial_number", trial.number)

        # モデルの学習と交差検証
        clf = RandomForestClassifier(**params)
        cv_scores = cross_val_score(
            clf, X_train, y_train, cv=3, scoring="accuracy"
        )
        score = cv_scores.mean()

        # メトリクスをロギング
        mlflow.log_metric("cv_accuracy_mean", score)
        mlflow.log_metric("cv_accuracy_std", cv_scores.std())

        # モデルをアーティファクトとして保存
        clf.fit(X_train, y_train)
        test_accuracy = clf.score(X_test, y_test)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.sklearn.log_model(clf, "model")

    return score

print("目的関数を定義しました")

## セクション 3: Optuna による最適化の実行

親 Run（`run_name="optuna_study"`）の下に 20 回の trial を実行します。

### 親子 Run の構造

```
optuna_study  (親 Run)
├── trial_0   (子 Run, nested=True)
├── trial_1   (子 Run, nested=True)
│   ...
└── trial_19  (子 Run, nested=True)
```

この構造により、MLflow UI で study 全体と individual trial の両方を管理できます。

In [ ]:
# 親 Run として study 全体を wrap
with mlflow.start_run(run_name="optuna_study") as parent_run:
    parent_run_id = parent_run.info.run_id

    # TPESampler: Tree-structured Parzen Estimator（再現性のため seed 固定）
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="breast_cancer_rf_hpo",
    )

    # 最適化の実行（n_trials=20）
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    # 親 Run にサマリーをロギング
    mlflow.log_metric("best_cv_accuracy", study.best_value)
    mlflow.log_param("n_trials", 20)
    mlflow.log_param("sampler", "TPESampler")
    mlflow.log_param("direction", "maximize")
    mlflow.log_params({
        f"best_{k}": v for k, v in study.best_trial.params.items()
    })

print(f"\n=== 最適化結果 ===")
print(f"Best trial: #{study.best_trial.number}")
print(f"Best value: {study.best_value:.4f}")
print(f"Best params: {study.best_trial.params}")
print(f"\n親 Run ID: {parent_run_id}")

## セクション 4: 最適化履歴の可視化

In [ ]:
import matplotlib.pyplot as plt

# 各 trial の結果を可視化
trial_numbers = [t.number for t in study.trials]
trial_values = [t.value for t in study.trials]
best_so_far = [max(trial_values[:i+1]) for i in range(len(trial_values))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trial ごとの CV accuracy
axes[0].plot(trial_numbers, trial_values, "o-", alpha=0.6, label="Trial accuracy")
axes[0].plot(trial_numbers, best_so_far, "r-", linewidth=2, label="Best so far")
axes[0].axhline(y=study.best_value, color="green", linestyle="--", alpha=0.7,
                label=f"Best: {study.best_value:.4f}")
axes[0].set_xlabel("Trial number")
axes[0].set_ylabel("CV Accuracy (mean)")
axes[0].set_title("Optimization History")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# パラメータの重要度（Optuna の feature importance）
importances = optuna.importance.get_param_importances(study)
param_names = list(importances.keys())
importance_values = list(importances.values())

axes[1].barh(param_names, importance_values, color="steelblue")
axes[1].set_xlabel("Importance")
axes[1].set_title("Parameter Importance")
axes[1].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("/tmp/optuna_study_results.png", dpi=100, bbox_inches="tight")
plt.show()
print("グラフを保存しました")

## セクション 5: mlflow.search_runs() による Run の検索と比較

`mlflow.search_runs()` を使うと、プログラム上から Run を検索・フィルタリング・ソートできます。

### 主なオプション

| オプション | 説明 |
|-----------|------|
| `experiment_names` | 対象 Experiment 名のリスト |
| `filter_string` | SQLライクなフィルタ条件 |
| `order_by` | ソート条件（ASC/DESC） |
| `max_results` | 取得する最大 Run 数 |

In [ ]:
# 子 Run（trial）のみを検索（parentRunId でフィルタ）
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.mlflow.parentRunId IS NOT NULL",
    order_by=["metrics.cv_accuracy_mean DESC"],
)

print(f"検索結果: {len(runs_df)} 件の trial Run")
print("\nTop 5 Run:")
display_cols = [
    "run_id",
    "metrics.cv_accuracy_mean",
    "metrics.cv_accuracy_std",
    "metrics.test_accuracy",
    "params.n_estimators",
    "params.max_depth",
    "params.max_features",
]
print(runs_df[display_cols].head().to_string(index=False))

In [ ]:
# ベスト Run の確認
best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_cv_accuracy = best_run["metrics.cv_accuracy_mean"]

print(f"ベスト Run ID: {best_run_id}")
print(f"CV Accuracy (mean): {best_cv_accuracy:.4f}")
print(f"CV Accuracy (std):  {best_run['metrics.cv_accuracy_std']:.4f}")
print(f"Test Accuracy:      {best_run['metrics.test_accuracy']:.4f}")
print(f"\nパラメータ:")
for col in ["params.n_estimators", "params.max_depth", "params.min_samples_split", "params.max_features"]:
    print(f"  {col.replace('params.', '')}: {best_run[col]}")

## セクション 6: MLflow UI の「Compare」機能

MLflow UI では複数の Run を視覚的に比較できます。

### UI での比較手順

1. ターミナルで `mlflow ui` を実行（または `mlflow server`）
2. ブラウザで `http://localhost:5000` を開く
3. `optuna_hpo` Experiment を選択
4. 比較したい Run のチェックボックスを複数選択
5. 「Compare」ボタンをクリック

### 確認できる項目

- **Parallel Coordinates**: 全パラメータとメトリクスの関係を並行座標で表示
- **Scatter Plot**: 任意の 2 つのメトリクス/パラメータの散布図
- **Box Plot**: 各パラメータの分布
- **Contour Plot**: 2 パラメータとメトリクスの等高線図

> **Tip**: 親 Run（`optuna_study`）を展開すると、全 trial の子 Run が一覧表示されます。

## セクション 7: ベストモデルの Model Registry への登録

`study.best_trial` からベストパラメータを取得し、対応する Run のモデルを
Model Registry に登録します。

In [ ]:
# Optuna の best_trial と MLflow の検索結果を照合
best_trial_number = study.best_trial.number
print(f"Optuna best trial: #{best_trial_number}")
print(f"Optuna best value: {study.best_value:.4f}")

# trial_number パラメータで対応する Run を特定
trial_run = runs_df[runs_df["params.trial_number"] == str(best_trial_number)]
if len(trial_run) == 0:
    # フォールバック: cv_accuracy_mean でベスト Run を使用
    print("trial_number による照合に失敗。cv_accuracy_mean でベスト Run を使用。")
    matched_run_id = best_run_id
else:
    matched_run_id = trial_run.iloc[0]["run_id"]
    print(f"照合成功: Run ID = {matched_run_id}")

print(f"\n登録対象 Run ID: {matched_run_id}")

In [ ]:
# Model Registry にベストモデルを登録
MODEL_NAME = "BreastCancerClassifier"
model_uri = f"runs:/{matched_run_id}/model"

print(f"モデル URI: {model_uri}")
print(f"登録先モデル名: {MODEL_NAME}")

registered = mlflow.register_model(model_uri, MODEL_NAME)
print(f"\n登録完了!")
print(f"  モデル名: {registered.name}")
print(f"  バージョン: {registered.version}")
print(f"  ステータス: {registered.status}")

In [ ]:
# 登録したモデルを読み込んでテストデータで推論
loaded_model = mlflow.sklearn.load_model(model_uri)
test_preds = loaded_model.predict(X_test)
test_accuracy = (test_preds == y_test).mean()
print(f"登録モデルのテスト精度: {test_accuracy:.4f}")

## セクション 8: 結果の検証

In [ ]:
# Optuna の結果と MLflow 検索結果の整合性を確認
search_best_cv_accuracy = runs_df.iloc[0]["metrics.cv_accuracy_mean"]

print("=== 整合性チェック ===")
print(f"Optuna best_value:          {study.best_value:.6f}")
print(f"MLflow search best_value:   {search_best_cv_accuracy:.6f}")

# 小数点以下4桁の精度で一致することを確認
assert abs(study.best_value - search_best_cv_accuracy) < 1e-4, (
    f"不一致: Optuna={study.best_value:.6f}, MLflow={search_best_cv_accuracy:.6f}"
)
print("\nアサーション通過: Optuna と MLflow の結果が一致しています")

# モデルの登録確認
client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
print(f"\n登録済みモデルバージョン数: {len(versions)}")
for v in versions:
    print(f"  Version {v.version}: run_id={v.run_id[:8]}...")

## まとめ

このノートブックでは以下を学びました:

### Optuna + MLflow 統合のポイント

1. **親子 Run 構造**: `mlflow.start_run(nested=True)` で study 全体を親 Run、各 trial を子 Run として管理
2. **探索空間の定義**:
   - `trial.suggest_int(name, low, high)` - 整数パラメータ
   - `trial.suggest_float(name, low, high)` - 連続値パラメータ
   - `trial.suggest_categorical(name, choices)` - カテゴリパラメータ
3. **TPESampler**: ベイズ最適化ベースのサンプラー。`seed=42` で再現性を確保

### Run の検索と比較

4. **`mlflow.search_runs()`**: `filter_string` でフィルタ、`order_by` でソートして Run を取得
5. **MLflow UI の Compare 機能**: 複数 Run を選択して並行座標・散布図・箱ひげ図で視覚的に比較

### モデル管理

6. **`mlflow.register_model()`**: ベスト Run のモデルを Model Registry に登録
7. 登録後は `models:/BreastCancerClassifier@champion` 形式でエイリアスを使ってロード可能

---

**次のステップ**: [05_champion_challenger](../05_champion_challenger/) でチャンピオン/チャレンジャー管理とモデルの昇格・ロールバック手順を学びます。